### MultiIndexing
MultiIndex lets you create hierarchical row/column indexes to represent grouped or nested data.

In [2]:
import numpy as np
import pandas as pd

### Series 1D and DataFrame are 2D objects
Series are 1-D labeled arrays; DataFrame is a 2-D table made of Series. Index provides labels for alignment and selection.

In [3]:
# can have multiple indices; let's try
index_val=[('cse',2019),('cse',2020),('cse',2021),('cse',2022),('cse',2023),('cse',2024)]
a=pd.Series([1,2,3,4,5,6],index=index_val)
a

(cse, 2019)    1
(cse, 2020)    2
(cse, 2021)    3
(cse, 2022)    4
(cse, 2023)    5
(cse, 2024)    6
dtype: int64

## How to create MultiIndex objects
Build MultiIndex using constructors from tuples or by taking cartesian products of level values.

### pd.MultiIndex.from_tuples()
Create a MultiIndex from a list of `(level1, level2, ...)` tuples.

In [4]:
index_val=[('cse',2019),('cse',2020),('cse',2021),('ece',2019),('ece',2020),('ece',2021)]
multiindex=pd.MultiIndex.from_tuples(index_val)
multiindex.levels[1]

Index([2019, 2020, 2021], dtype='int64')

### pd.MultiIndex.from_product()
Generate a MultiIndex from the cartesian product of iterables (useful to create full grid of levels).

In [5]:
pd.MultiIndex.from_product([['cse','ece'],[2019,2021,2022,2023,2024]])

MultiIndex([('cse', 2019),
            ('cse', 2021),
            ('cse', 2022),
            ('cse', 2023),
            ('cse', 2024),
            ('ece', 2019),
            ('ece', 2021),
            ('ece', 2022),
            ('ece', 2023),
            ('ece', 2024)],
           )

### Create a Series with multiIndex objects
Attach a MultiIndex to a `Series` to store hierarchical indexed values (example: branch, year).

In [6]:
s=pd.Series([1,2,3,4,5,6],index=multiindex)
s

cse  2019    1
     2020    2
     2021    3
ece  2019    4
     2020    5
     2021    6
dtype: int64

### How to fetch data such as series
Use label-based selection (e.g., `s['cse']`) or `loc` with tuples to access specific subgroups.

In [7]:
s['cse']

2019    1
2020    2
2021    3
dtype: int64

### MultiIndex to DataFrame using unstack
`unstack()` pivots an inner index level into columns producing a regular DataFrame.

In [8]:
temp=s.unstack()
temp

,2019,2020,2021
cse,1,2,3
ece,4,5,6


### DataFrame to MultiIndex using stack
`stack()` pivots columns into the innermost row index level, producing a Series with MultiIndex.

In [9]:
temp.stack()

cse  2019    1
     2020    2
     2021    3
ece  2019    4
     2020    5
     2021    6
dtype: int64

### MultiIndex DataFrame
Construct DataFrames with MultiIndex in rows and/or columns to model hierarchical data (example included).

In [10]:
branch_df= pd.DataFrame(
    [
    [1,2],
    [3,4],
    [5,6],
    [7,8],
    [9,10],
    [11,12],
    ],
    index=multiindex,
    columns=['avg_package','students']
)
branch_df

avg_package  students
cse 2019            1         2
    2020            3         4
    2021            5         6
ece 2019            7         8
    2020            9        10
    2021           11        12

In [11]:
branch_df.iloc[4]
branch_df.loc['ece']

,avg_package,students
2019,7,8
2020,9,10
2021,11,12


### MultiIndex DataFrame from column perspective
Columns can be hierarchical as well; access subcolumns like `df['delhi']['avg_package']`.

In [12]:
branch_df2=pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
    ],
    index=[2019,2021,2022,2023],
    columns=pd.MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])
)

branch_df2

delhi               mumbai         
     avg_package students avg_package students
2019           1        2           0        0
2021           3        4           0        0
2022           5        6           0        0
2023           7        8           0        0

In [13]:
branch_df2['delhi']['avg_package']

2019    1
2021    3
2022    5
2023    7
Name: avg_package, dtype: int64

### MultiIndex DataFrame in terms of both cols and index
You can combine row and column MultiIndexes to represent complex table layouts.

In [14]:
branch_df3=pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
        [9,10,0,0],
        [11,12,0,0],
    ],
    index=multiindex,
    columns=pd.MultiIndex.from_product([['Delhi','Mumbai'],['Avg_package','Student']])
)
branch_df3

Delhi              Mumbai        
         Avg_package Student Avg_package Student
cse 2019           1       2           0       0
    2020           3       4           0       0
    2021           5       6           0       0
ece 2019           7       8           0       0
    2020           9      10           0       0
    2021          11      12           0       0

### Stacking and Unstacking
Use `stack()` / `unstack()` to reshape between long and wide representations when hierarchies exist.

- stack()
- unstack()

In [15]:
branch_df3.unstack()

Delhi                                  Mumbai                    \
    Avg_package           Student           Avg_package           Student   
           2019 2020 2021    2019 2020 2021        2019 2020 2021    2019   
cse           1    3    5       2    4    6           0    0    0       0   
ece           7    9   11       8   10   12           0    0    0       0   

               
               
    2020 2021  
cse    0    0  
ece    0    0

In [16]:
branch_df3.stack()

C:\Users\Neelesh Ranpuriya\AppData\Local\Temp\ipykernel_18108\4148153360.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  branch_df3.stack()


Delhi  Mumbai
cse 2019 Avg_package      1       0
         Student          2       0
    2020 Avg_package      3       0
         Student          4       0
    2021 Avg_package      5       0
         Student          6       0
ece 2019 Avg_package      7       0
         Student          8       0
    2020 Avg_package      9       0
         Student         10       0
    2021 Avg_package     11       0
         Student         12       0

### Working With Multiindex dataframes
Common checks: `head()`, `shape`, `info()`, `isna()`, `duplicated()` to inspect and clean hierarchical data.

In [17]:
# head and tail
branch_df3.head()
# Shape
branch_df3.shape
# info
branch_df3.unstack().info()
# Duplicate -> is null
branch_df3.duplicated().sum()
# Null values
branch_df3.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 2 entries, cse to ece
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   (Delhi, Avg_package, 2019)   2 non-null      int64
 1   (Delhi, Avg_package, 2020)   2 non-null      int64
 2   (Delhi, Avg_package, 2021)   2 non-null      int64
 3   (Delhi, Student, 2019)       2 non-null      int64
 4   (Delhi, Student, 2020)       2 non-null      int64
 5   (Delhi, Student, 2021)       2 non-null      int64
 6   (Mumbai, Avg_package, 2019)  2 non-null      int64
 7   (Mumbai, Avg_package, 2020)  2 non-null      int64
 8   (Mumbai, Avg_package, 2021)  2 non-null      int64
 9   (Mumbai, Student, 2019)      2 non-null      int64
 10  (Mumbai, Student, 2020)      2 non-null      int64
 11  (Mumbai, Student, 2021)      2 non-null      int64
dtypes: int64(12)
memory usage: 208.0+ bytes


Delhi   Avg_package    0
        Student        0
Mumbai  Avg_package    0
        Student        0
dtype: int64

### Extract Rows & columns
Use `loc` with tuple keys for label selection and `iloc` for positional selection across levels.

In [18]:
branch_df3.loc['cse',2019]

Delhi   Avg_package    1
        Student        2
Mumbai  Avg_package    0
        Student        0
Name: (cse, 2019), dtype: int64

In [19]:
# Multiple Rows
branch_df3.loc[('cse',2019):('ece',2020):2]

Delhi              Mumbai        
         Avg_package Student Avg_package Student
cse 2019           1       2           0       0
    2021           5       6           0       0
ece 2020           9      10           0       0

In [20]:
# iloc
branch_df3.iloc[0:5:2]

Delhi              Mumbai        
         Avg_package Student Avg_package Student
cse 2019           1       2           0       0
    2021           5       6           0       0
ece 2020           9      10           0       0

In [21]:
# Single columns
branch_df3['Delhi']['Student'].loc['cse']

2019    2
2020    4
2021    6
Name: Student, dtype: int64

In [22]:
# Multiple Columns
branch_df3.iloc[:,1:3]

Delhi      Mumbai
         Student Avg_package
cse 2019       2           0
    2020       4           0
    2021       6           0
ece 2019       8           0
    2020      10           0
    2021      12           0

In [23]:
# extract both
branch_df3.iloc[[0,4],[0,3]]


,,Delhi,Mumbai
,,Avg_package,Student
cse,2019,1,0
ece,2020,9,0


In [24]:
# Sorting Index
branch_df3.sort_index(ascending=False)
branch_df3.sort_index(ascending=[False,True])
branch_df3.sort_index(level=1,ascending=False)

Delhi              Mumbai        
         Avg_package Student Avg_package Student
ece 2021          11      12           0       0
cse 2021           5       6           0       0
ece 2020           9      10           0       0
cse 2020           3       4           0       0
ece 2019           7       8           0       0
cse 2019           1       2           0       0

In [25]:
# Transpose
branch_df3.transpose()

cse            ece          
                   2019 2020 2021 2019 2020 2021
Delhi  Avg_package    1    3    5    7    9   11
       Student        2    4    6    8   10   12
Mumbai Avg_package    0    0    0    0    0    0
       Student        0    0    0    0    0    0

In [26]:
# Swap level
branch_df3.swaplevel(axis=1)

Avg_package Student Avg_package Student
               Delhi   Delhi      Mumbai  Mumbai
cse 2019           1       2           0       0
    2020           3       4           0       0
    2021           5       6           0       0
ece 2019           7       8           0       0
    2020           9      10           0       0
    2021          11      12           0       0

### Long Vs Wild Data
Convert wide (wild) tables to long format using `melt()` for easier analysis and grouping.

- Melt -> Simple example of branch
- Wild Data To Long Data

In [27]:
pd.DataFrame({'cse':[120,120],'ese':[100,100],'mac':[150,150]}).melt(var_name='Branch',value_name='Students')

,Branch,Students
0,cse,120
1,cse,120
2,ese,100
3,ese,100
4,mac,150
5,mac,150


In [28]:
pd.DataFrame(
    {
        'branch':['cse','ece','mech'],
        '2020':[100,150,60],
        '2021':[120,130,80],
        '2022':[150,140,70]
    }
).melt(id_vars=['branch'],var_name='Year',value_name='Students')

,branch,Year,Students
0,cse,2020,100
1,ece,2020,150
2,mech,2020,60
3,cse,2021,120
4,ece,2021,130
5,mech,2021,80
6,cse,2022,150
7,ece,2022,140
8,mech,2022,70


In [29]:
# Real World Example
death=pd.read_csv('Datasets/time_series_covid19_deaths_global.csv')
Confirm=pd.read_csv('Datasets/time_series_covid19_confirmed_global.csv')
Confirm.head(5)

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,12/24/22,12/25/22,12/26/22,12/27/22,12/28/22,12/29/22,12/30/22,12/31/22,1/1/23,1/2/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,207310,207399,207438,207460,207493,207511,207550,207559,207616,207627
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,333749,333749,333751,333751,333776,333776,333806,333806,333811,333812
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271194,271198,271198,271202,271208,271217,271223,271228,271229,271229
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47686,47686,47686,47686,47751,47751,47751,47751,47751,47751
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,104973,104973,104973,105095,105095,105095,105095,105095,105095,105095


In [30]:
death.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,12/24/22,12/25/22,12/26/22,12/27/22,12/28/22,12/29/22,12/30/22,12/31/22,1/1/23,1/2/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,7845,7846,7846,7846,7846,7847,7847,7849,7849,7849
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,3595,3595,3595,3595,3595,3595,3595,3595,3595,3595
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,6881,6881,6881,6881,6881,6881,6881,6881,6881,6881
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,165,165,165,165,165,165,165,165,165,165
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,1928,1928,1928,1930,1930,1930,1930,1930,1930,1930


In [31]:
death=death.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='Date',value_name='no_of_death')

In [32]:
Confirm = Confirm.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='Date',value_name='num_case')

In [53]:
Corona=death.merge(Confirm,how='inner',on=['Province/State','Country/Region','Lat','Long','Date'])[['Country/Region','num_case','no_of_death']]

In [34]:
Corona

,Country/Region,num_case,no_of_death
0,Afghanistan,0,0
1,Albania,0,0
2,Algeria,0,0
3,Andorra,0,0
4,Angola,0,0
...,...,...,...
311248,West Bank and Gaza,703228,5708
311249,Winter Olympics 2022,535,0
311250,Yemen,11945,2159
311251,Zambia,334661,4024


In [68]:
Corona[Corona['Country/Region']=='China']['no_of_death'].max()

np.int64(11943)

In [71]:
no_of_death_per_country=Corona.groupby('Country/Region')['no_of_death'].max().sort_values(ascending=False)

In [72]:
no_of_cases_per_country=Corona.groupby('Country/Region')['num_case'].max().sort_values(ascending=False)

In [73]:
# Covid-19 cases and death percentage
percentage_ratio = (no_of_death_per_country / no_of_cases_per_country.replace(0, pd.NA)) * 100
percentage_ratio.sort_values(ascending=False)

Country/Region
Korea, North            600.000000
MS Zaandam               22.222222
Yemen                    18.074508
Sudan                     7.841598
Syria                     5.508246
                           ...    
Antarctica                0.000000
Holy See                  0.000000
Summer Olympics 2020      0.000000
Tuvalu                    0.000000
Winter Olympics 2022      0.000000
Length: 201, dtype: float64

In [39]:
# Day 6 - Functions / methods used (for quick reference)
# pd.MultiIndex.from_tuples()
# pd.MultiIndex.from_product()
# pd.Series / pd.DataFrame constructors
# Series.unstack()
# DataFrame.stack()
# DataFrame.unstack()
# DataFrame.sort_index()
# DataFrame.swaplevel()
# DataFrame.transpose()
# DataFrame.melt()
# pd.read_csv()
# DataFrame.merge()
# DataFrame.groupby()
# Series.max()
# Series.sort_values()
# DataFrame.isna()
# DataFrame.duplicated()
# DataFrame.head() / .tail()
# iloc / loc selectors
# .info()
# (End of Day 6 reference)
